# Predicting Air Quality Using Machine Learning

## Project Overview

This project aims to analyze air quality data in India to gain insights into pollution levels, their spatial and temporal variations, and potential contributing factors. By examining trends and patterns in air quality measurements, the project seeks to understand the impact of pollution on public health and the environment and inform strategies for pollution mitigation and environmental policy.

## Approach & Steps

We will follow these steps to conduct our analysis:

1. Problem Definition
2. Data Definition
3. Data Exploration
4. Data Preprocessing
5. Feature Engineering
6. Model Selection
7. Model Training
8. Model Evaluation
9. Hyperparameter Tuning
10. Model Interpretation

**Dataset Link:** [Air Quality Data in India](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)

---

## 1. Problem Definition

This problem encompasses tasks such as data collection, preprocessing, exploratory data analysis, feature engineering, statistical analysis, machine learning modeling, and visualization. The ultimate goal is to leverage data-driven insights to address challenges related to air pollution and promote sustainable development and public well-being in India.

---

## 2. Data Definition

Below are the key variables in the dataset:

### 1. City

- The name of the city where the air quality measurements were recorded.
- Cities may vary in size, population density, industrial activity, traffic volume, and geographical location, all of which can influence air pollution levels.
- Categorical variable. Example values: "Delhi", "Mumbai", "Bangalore", "Chennai".

### 2. Date

- The date on which the air quality measurements were taken.
- Time-series analysis of air quality data can reveal seasonal variations, trends over time, and short-term fluctuations in pollution levels.
- Temporal variable. Example values: "2022-01-01", "2022-01-02".

### 3. PM2.5 (Particulate Matter 2.5)

- Fine particulate matter with a diameter of 2.5 micrometers or less. These particles are small enough to penetrate deep into the lungs and can cause respiratory and cardiovascular health issues.
- Sources: vehicle emissions, industrial processes, construction activities, and biomass burning.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 4. PM10 (Particulate Matter 10)

- Inhalable coarse particles with a diameter of 10 micrometers or less.
- Sources: dust, pollen, road dust, and construction activities.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 5. NO (Nitric Oxide)

- A colorless gas produced by combustion processes, such as vehicle engines and industrial operations.
- Can react with other pollutants in the atmosphere to form nitrogen dioxide (NO2) and contribute to the formation of ground-level ozone and fine particulate matter.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 6. NO2 (Nitrogen Dioxide)

- A reddish-brown gas that forms from the oxidation of nitric oxide (NO) in the atmosphere.
- Primarily emitted from vehicle exhaust, industrial facilities, and power plants.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 7. NOx (Nitrogen Oxides)

- A group of reactive gases that include nitric oxide (NO) and nitrogen dioxide (NO2).
- Produced from combustion processes, particularly those involving high temperatures.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 8. NH3 (Ammonia)

- A colorless gas commonly used in agricultural fertilizers and industrial processes.
- Also emitted from livestock operations and wastewater treatment plants.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 9. CO (Carbon Monoxide)

- A colorless, odorless gas produced by incomplete combustion of fossil fuels, biomass, and other organic materials.
- Toxic at high concentrations and can impair oxygen transport in the bloodstream.
- Continuous variable (ppm or mg/m³). Typical range: 0 to several ppm.

### 10. SO2 (Sulfur Dioxide)

- A colorless gas with a pungent odor produced by burning fossil fuels containing sulfur.
- A major air pollutant emitted from industrial processes, power plants, and vehicle engines.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 11. O3 (Ozone)

- A colorless gas occurring naturally in the Earth's upper atmosphere and also formed in the lower atmosphere.
- Ground-level ozone is a key component of smog and can cause respiratory problems.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 12. Benzene

- A colorless, flammable liquid with a sweet odor used as a solvent in industrial processes.
- Found in motor vehicle exhaust, cigarette smoke, and certain consumer products.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 13. Toluene

- A colorless liquid with a sweet, pungent odor used in paints, coatings, adhesives, and industrial products.
- Found in motor vehicle exhaust and tobacco smoke.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 14. Xylene

- A colorless liquid with a sweet, aromatic odor used as a solvent in industrial products.
- Found in motor vehicle exhaust and cigarette smoke.
- Continuous variable (µg/m³). Typical range: 0 to several hundred µg/m³.

### 15. AQI (Air Quality Index)

- A standardized index used to communicate the quality of the air and associated health risks.
- Calculated based on the concentrations of various pollutants such as PM2.5, PM10, NO2, SO2, CO, and O3.
- Discrete variable, range: 0 to 500.

### 16. AQI_Bucket

- Categorical variable representing the classification of AQI into predefined buckets.
- Categories: "Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe".

---


In [ ]:
# import shap
import warnings
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from scipy import interpolate
from matplotlib import pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")
np.random.seed(0)
%matplotlib inline

In [ ]:
df = pd.read_csv('city_hour.csv', parse_dates = ['Datetime'], low_memory = False)

In [ ]:
df.info()

In [ ]:
# Set the display format for float values
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
df.describe()

In [ ]:
df.head()

### **Missing values**


In [ ]:
df["Date"] = df.Datetime.dt.date.astype(str)
df.Datetime = df.Datetime.astype(str)

In [ ]:
df["PM10"] = df.groupby("City")["PM10"].rolling(window = 24, min_periods = 16).mean().values
df["PM2.5"] = df.groupby("City")["PM2.5"].rolling(window = 24, min_periods = 16).mean().values
df["SO2"] = df.groupby("City")["SO2"].rolling(window = 24, min_periods = 16).mean().values
df["CO"] = df.groupby("City")["CO"].rolling(window = 8, min_periods = 1).max().values
df["O3"] = df.groupby("City")["O3"].rolling(window = 8, min_periods = 1).max().values
df["NO2"] = df.groupby("City")["O3"].rolling(window = 24, min_periods = 1).max().values

In [ ]:
round(df.isna().sum()/len(df) * 100, 2)

### **AQI_Bucket category count**


In [ ]:
aqi_bucket_percentages = (df['AQI_Bucket'].value_counts() / len(df)) * 100
plt.figure(figsize=(8, 8))
plt.pie(aqi_bucket_percentages, labels=aqi_bucket_percentages.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of AQI Buckets')
plt.show()

In [ ]:
df.groupby('AQI_Bucket').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
#AQI_Bucket with respect to cities
plt.figure(figsize=(12,6))
sns.histplot(data=df,x='City',hue='AQI_Bucket',palette='Dark2',multiple='stack',shrink=0.8)
plt.xticks(rotation=90)
plt.show()

In [ ]:
def pair_plot(df):
  sns.pairplot(df, diag_kind="kde", markers="+",
                  plot_kws=dict(s=50, edgecolor="b", linewidth=1),
                  diag_kws=dict(shade=True))

In [ ]:
# pair_plot(df)

In [ ]:
def column_density(df, columns):
    n_rows = len(columns)
    fig, axes = plt.subplots(nrows=n_rows, ncols=1, figsize=(10, 10))

    for i, col in enumerate(columns):
        sns.kdeplot(data=df, x=col,color='red', fill=True, ax=axes[i])

    plt.tight_layout()
    plt.show()


In [ ]:
col = ['PM2.5','PM10','NO2','CO','O3','SO2','AQI']
column_density(df,col)

In [ ]:
def outlier_detection(df,columns):
    n_rows = len(columns)
    fig, axes = plt.subplots(nrows=n_rows, ncols=1, figsize=(10, 25))

    for i, col in enumerate(columns):
        sns.boxplot(data=df,x='AQI_Bucket',y=col,palette='Dark2', fill=True, ax=axes[i])
    plt.show()


In [ ]:
col = ['PM2.5','PM10','NO2','CO','O3','SO2','AQI']
outlier_detection(df,col)

### **Filling missing values**


In [ ]:
df.isna().sum()

In [ ]:
df = df.drop(['NO', 'NOx', 'NH3', 'Benzene', 'Toluene', 'Xylene'] , axis = 1)

### **Correlation matrix**


In [ ]:
def plot_corr_matrix(df):
    # Create the correlation matrix
    numeric_columns = df.select_dtypes(include=['float64']).columns
    plt.figure(figsize=(12,6))
    correlation_matrix = df[numeric_columns].corr()

    # Generate a heatmap
    sns.heatmap(correlation_matrix, annot=True)
    plt.show()

In [ ]:
plot_corr_matrix(df)

As we can see in correlation matrix that benzene,toluene and xylene are not contributing so much in predicting AQI.


In [ ]:
df.dropna(subset =['AQI_Bucket'], inplace = True)
df.sort_values(by = ['City', 'AQI_Bucket', 'Date'], inplace = True)

In [ ]:
df.shape

In [ ]:
df1 = df.copy()

In [ ]:
df1.isna().sum()

In [ ]:
# Exclude columns not suitable for filling missing values
columns_to_exclude = ['Date', 'Datetime', 'City', 'AQI_Bucket', 'AQI']
columns_to_fill = [col for col in df1.columns if col not in columns_to_exclude]

# Fill missing values with group-specific mean values within each group
df1_grouped_mean = df1.groupby('AQI_Bucket')[columns_to_fill].transform(lambda x: x.fillna(x.mean()))

# Update the original DataFrame with filled values
df1[columns_to_fill] = df1_grouped_mean

# Reset the index
df1.reset_index(drop=True, inplace=True)

In [ ]:
df1.sort_values(by = ['City', 'AQI_Bucket', 'Datetime'], inplace = True)
df1.head()

In [ ]:
# For DataFrame df
df_selection = df[(df['City'] == 'Ahmedabad') & (df['AQI_Bucket'] == 'Moderate')]

# For DataFrame df1
df1_selection = df1[(df1['City'] == 'Ahmedabad') & (df1['AQI_Bucket'] == 'Moderate')]

In [ ]:
df_selection.isna().sum()

In [ ]:
len(df_selection)

In [ ]:
df1_selection.isna().sum()

In [ ]:
df1.isna().sum()

In [ ]:
print(df_selection)
print(df1_selection)

In [ ]:
col = ['PM2.5','PM10','NO2','CO','O3','SO2']
column_density(df1,col)

In [ ]:
plot_corr_matrix(df1)

In [ ]:
pair_plot(df1)

In [ ]:
df1.describe()

In [ ]:
df1.info()

In [ ]:
df1.shape

In [ ]:
df2 = df1[df1['AQI'] <= 600]

In [ ]:
df2.shape

In [ ]:
df2.describe()

In [ ]:
column_density(df2, col)

In [ ]:
plot_corr_matrix(df2)

In [ ]:
df1.to_csv('no_missing.csv', index = False)

# **OUTLIERS DETECT & REMOVE**


In [ ]:
df_no_missing = pd.read_csv('no_missing.csv', parse_dates = ['Date'], low_memory = False)

In [ ]:
df_no_missing02 = df_no_missing.copy() #data frame where the ouliers will be removed

In [ ]:
buckets = df.AQI_Bucket.unique()
columns = ['PM2.5', 'PM10', 'NO', 'NO2', 'CO', 'SO2', 'O3']
print(buckets)

In [ ]:
#function for calculating lower and upper bounds for each column using percentiles

def evaluate_limits(df, aqi_bucket, single_column):
    # Get the data for the current aqi_bucket
    data = df[df['AQI_Bucket'] == aqi_bucket]

    lower_percentile = 5
    upper_percentile = 95
    lower_bound = data[single_column].quantile(lower_percentile / 100)
    upper_bound = data[single_column].quantile(upper_percentile / 100)

    return lower_bound, upper_bound

In [ ]:
from tabulate import tabulate

def show_limits(df, buckets, columns):
      for aqi_bucket in buckets:
        table_data = []  # List to store table rows

        # Get the data for the current aqi_bucket
        data = df[df['AQI_Bucket'] == aqi_bucket]

        for column in columns:
            # Getting upper and lower bound from calling functions
            lower_bound, upper_bound = evaluate_limits(df, aqi_bucket, column)
            table_data.append([column, lower_bound, upper_bound])

            # Print the table using tabulate with pipe format
        print(f"AQI Bucket: {aqi_bucket}")
        print("-" * 44)
        print(tabulate(table_data, headers=['Column', 'Lower Bound', 'Upper Bound'], tablefmt='pipe'))
        print("-" * 44)
        print()
        print()

In [ ]:
#show limits before removal of outliers
columns = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3', 'SO2']
buckets = ['Severe', 'Poor','Very Poor', 'Moderate','Satisfactory','Good']
show_limits(df_no_missing, buckets, columns)

In [ ]:
#print first 3 outliers of some columns

def show_outliers(df, buckets, specific_columns):
    for aqi_bucket in buckets:
        for column in specific_columns:
            lower_bound, upper_bound = evaluate_limits(df, aqi_bucket, column)
            print(f"Column: {column}   AQI Bucket: {aqi_bucket}")
            outliers = df[(df[column] > upper_bound) | (df[column] < lower_bound)].head(3)
            if not outliers.empty:
                # Specify the columns to show the spcific ones
                outliers_filtered = outliers[specific_columns]
                print(outliers_filtered)
            else:
                print("No outliers found.")
            print()
            print()
            print()


specific_columns = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3', 'SO2']
buckets = ['Severe', 'Poor','Very Poor', 'Moderate','Satisfactory','Good']
show_outliers(df_no_missing, buckets, specific_columns)


In [ ]:
def replace_outliers(df, buckets, columns):
    for aqi_bucket in buckets:
        # Get the data for the current label
        data = df[df['AQI_Bucket'] == aqi_bucket]

        # getting lower and upper bounds for each column by calling predefined function
        for column in columns:
            lower_bound, upper_bound = evaluate_limits(df, aqi_bucket, column)

            # Find the outlier indices
            outlier_indices = data[((data[column] > upper_bound) | (data[column] < lower_bound))].index

            # Replace outliers with the mean
            df.loc[outlier_indices, column] = data[column].mean()
    return df


In [ ]:
buckets = ['Severe', 'Poor','Very Poor', 'Moderate','Satisfactory','Good']
columns = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3', 'SO2']

#removinf the outliers from dataframe df_no_missing02
df_no_missing02 = replace_outliers(df_no_missing02, buckets, columns)

In [ ]:
#print limits after removal of outliers
columns = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3', 'SO2']
buckets = ['Severe', 'Poor','Very Poor', 'Moderate','Satisfactory','Good']
show_limits(df_no_missing02, buckets, columns)

In [ ]:
#boxplot before removal of outliers
outlier_detection(df_no_missing, columns)

In [ ]:
#boxplot after removal of outliers
outlier_detection(df_no_missing02, columns)

In [ ]:
df_no_missing.describe()

In [ ]:
df_no_missing02.describe()

In [ ]:
df_no_missing02.isna().sum()

In [ ]:
#plotting without AQI_BUCKET for visualizing the whole data
def boxplt(df, columns):
    fig, axes = plt.subplots(1, len(columns), figsize=(15, 4))  # Adjust figsize as needed
    for idx, col in enumerate(columns):
        sns.boxplot(y=df[col], ax=axes[idx])
        axes[idx].set_xlabel(col)
        axes[idx].set_ylim(0, 600)  # Set the y-axis range from 0 to 300
    plt.tight_layout()
    plt.show()

columns = ['PM2.5', 'PM10', 'NO2', 'CO', 'O3', 'SO2']
print("Normal Boxplot beore removal of Outliers")
print()
boxplt(df_no_missing, columns)
print("\n\n")
print("Normal Boxplot after removal of Outliers")
print()
boxplt(df_no_missing02 , columns)

In [ ]:
  # Create the correlation matrix
print("Correlation matrix before removal of outliers")
print()
plot_corr_matrix(df_no_missing)

In [ ]:
# Create the correlation matrix
print("Correlation matrix before removal of outliers")
print()
plot_corr_matrix(df_no_missing02)

In [ ]:
col = ['PM2.5','PM10','NO2','CO','O3','SO2','AQI']
column_density(df_no_missing02,col)

In [ ]:
pair_plot(df_no_missing02)

In [ ]:
df_no_missing02.to_csv('clean_data.csv', index = False)

In [ ]:
df_no_missing02.isna().sum()

# Feature Engineering

1. Feature Encode
2. Train and Test Split
3. Rolling Average (Feature Creation)
4. Feature Scaling

Steps followed:

1. First encode the city
2. Split the encoded dataframe into train and test dataframes
3. Calculate the rolling average based on 7 day window with minimum window size of 1.
4. Drop the city and aqi_rw_avg
5. We can scale or leave the data as it is on demand


In [ ]:
from sklearn.covariance import EllipticEnvelope

# Read the clean data into a DataFrame
df_clean = pd.read_csv('clean_data.csv', low_memory=False)

# Extract other columns as DataFrame
other_cols = df_clean[['City', 'Datetime', 'AQI_Bucket']]

# Extract features (X) and target variable (y) as NumPy arrays
X = df_clean[['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3']].values
y = df_clean['AQI'].values

# Initialize Elliptic Envelope model
elliptic_envelope = EllipticEnvelope(contamination=0.10)  # Adjust contamination parameter as needed

# Fit the model to the data
elliptic_envelope.fit(np.column_stack((X, y)))

# Predict outliers
outlier_predictions = elliptic_envelope.predict(np.column_stack((X, y)))

# Identify outliers
outliers_indices = np.where(outlier_predictions == -1)[0]

# Remove outliers from X, y, and other_cols
X_filtered = np.delete(X, outliers_indices, axis=0)
y_filtered = np.delete(y, outliers_indices)
other_cols_filtered = other_cols.drop(outliers_indices)

# Now X_filtered, y_filtered, and other_cols_filtered contain the dataset with outliers removed
# Create DataFrame from X_filtered and y_filtered
df_filtered = pd.DataFrame(X_filtered, columns=['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3'])
df_filtered['AQI'] = y_filtered

# Reset index of other_cols_filtered and concatenate with df_filtered
df_filtered = pd.concat([df_filtered, other_cols_filtered.reset_index(drop=True)], axis=1)

In [ ]:
df_filtered.isna().sum()

In [ ]:
print(len(df_filtered))
print(len(other_cols))

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('PM2.5')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 0], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('PM2.5')
plt.ylabel('AQI')
plt.legend()

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 1], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('PM10')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 1], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('PM10')
plt.ylabel('AQI')
plt.legend()

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 2], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('NO2')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 2], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('NO2')
plt.ylabel('AQI')
plt.legend()

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 3], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('CO')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 3], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('CO')
plt.ylabel('AQI')
plt.legend()

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 4], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('O3')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 4], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('O3')
plt.ylabel('AQI')
plt.legend()

In [ ]:
# Create scatter plot of original data
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 5], y, color='blue', label='Original Data')
plt.title('Original Data')
plt.xlabel('SO2')
plt.ylabel('AQI')
plt.legend()

# Create scatter plot of filtered data
plt.subplot(1, 2, 2)
plt.scatter(X_filtered[:, 5], y_filtered, color='red', label='Filtered Data')
plt.title('Filtered Data (Outliers Removed)')
plt.xlabel('SO2')
plt.ylabel('AQI')
plt.legend()

In [ ]:
plot_corr_matrix(df_filtered)

In [ ]:
column_density(df_filtered, columns)

In [ ]:
df_filtered.columns

In [ ]:
df_filtered.isna().sum()

In [ ]:
column_order = ['Datetime', 'City', 'PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'O3', 'AQI_Bucket', 'AQI']
df_filtered = df_filtered.reindex(columns=column_order)

In [ ]:
df_filtered.to_csv('clean_filtered_data.csv', index = False)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory = False)
df_filter.head()

In [ ]:
aqi_bucket_percentages = (df_filtered['AQI_Bucket'].value_counts() / len(df_filtered)) * 100
plt.figure(figsize=(8, 8))
plt.pie(aqi_bucket_percentages, labels=aqi_bucket_percentages.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of AQI Buckets')
plt.show()

In [ ]:
def plot_regression_diagnostic_plots(y_train, y_train_pred, y_test, y_test_pred):

    train_residual = y_train - y_train_pred
    test_residual = y_test - y_test_pred
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))

    # Scatter Plot of Actual vs. Predicted Values
    sns.scatterplot(x=y_test_pred, y=y_test, ax=axes[0, 0])
    axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
    axes[0, 0].set_title("Actual vs. Predicted")
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("Actual")

    # Residual Plot
    sns.scatterplot(x=y_test_pred, y=test_residual, ax=axes[0, 1])
    axes[0, 1].axhline(y=0, color='r', linestyle='--')
    axes[0, 1].set_title("Residual Plot")
    axes[0, 1].set_xlabel("Predicted")
    axes[0, 1].set_ylabel("Residuals")

    # KDE Plot of y_test and y_test_pred
    sns.kdeplot(y_test, label='True', ax=axes[1, 0])
    sns.kdeplot(y_test_pred, label='Predicted', ax=axes[1, 0])
    axes[1, 0].set_title("KDE Plot of True vs. Predicted")
    axes[1, 0].set_xlabel("Value")
    axes[1, 0].set_ylabel("Density")
    axes[1, 0].legend()

    # Histogram of Residuals
    sns.histplot(test_residual, ax=axes[1, 1], kde=True)
    axes[1, 1].set_title("Histogram of Residuals")
    axes[1, 1].set_xlabel("Residuals")

    # Q-Q Plot
    import scipy.stats as stats
    stats.probplot(test_residual, dist="norm", plot=axes[2, 0])
    axes[2, 0].set_title("Q-Q Plot")

    # Fitted vs. Residual Plot
    sns.kdeplot(train_residual, label='Train Diff', color='blue', linestyle='-', ax=axes[2,1])
    sns.kdeplot(test_residual, label='Test Diff', color='orange', linestyle='--', ax=axes[2,1])
    axes[2, 1].set_xlabel('Prediction Difference')
    axes[2, 1].set_ylabel('Density')
    axes[2, 1].set_title('Kernel Density Estimation of Prediction Differences')
    axes[2, 1].legend()

    # Show plots
    plt.tight_layout()
    plt.show()

In [ ]:
X, y = df_filter[['PM2.5', 'PM10']], df_filter['AQI']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, shuffle=False)
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
plot_regression_diagnostic_plots(y_train, y_pred_train, y_test, y_pred_test)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# Split the data into training and testing sets
X, y = df_filter[['PM2.5', 'PM10']], df_filter['AQI']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Train the Ridge regression model with regularization (L2 penalty)
alpha = 0.1  # Regularization strength, adjust as needed
ridge_model = Ridge(alpha=alpha)
ridge_model.fit(X_train, y_train)

# Evaluate the model
r2_score = ridge_model.score(X_test, y_test)
print("R-squared score:", r2_score)

# Make predictions
y_pred_train = ridge_model.predict(X_train)
y_pred_test = ridge_model.predict(X_test)

In [ ]:
plot_regression_diagnostic_plots(y_train, y_pred_train, y_test, y_pred_test)

In [ ]:
from sklearn.linear_model import ElasticNet

# Train the ElasticNet regression model with L2 penalty
elastic_net = ElasticNet(alpha=0.1, l1_ratio=0.5)  # Adjust alpha and l1_ratio as needed
elastic_net.fit(X_train, y_train)

# Evaluate the model
r2_score_en = elastic_net.score(X_test, y_test)
print("R-squared score (ElasticNet):", r2_score_en)

# Make predictions
y_pred_train = elastic_net.predict(X_train)
y_pred_test = elastic_net.predict(X_test)

In [ ]:
plot_regression_diagnostic_plots(y_train, y_pred_train, y_test, y_pred_test)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Split the data into training and testing sets
X, y = df_filter[['PM2.5', 'PM10']], df_filter['AQI']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Define the parameter grid
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}  # Values of alpha to try

# Initialize the Ridge regression model
ridge_model = Ridge()

# Create the GridSearchCV object
grid_search = GridSearchCV(ridge_model, param_grid, cv=5, scoring='r2')

# Perform grid search to find the best parameters
grid_search.fit(X_train, y_train)

# Get the best parameter
best_alpha = grid_search.best_params_['alpha']
print("Best alpha:", best_alpha)

# Train the Ridge regression model with the best parameters
best_ridge_model = Ridge(alpha=best_alpha)
best_ridge_model.fit(X_train, y_train)

# Evaluate the model
r2_score = best_ridge_model.score(X_test, y_test)
print("R-squared score:", r2_score)

# Make predictions
y_pred_train = best_ridge_model.predict(X_train)
y_pred_test = best_ridge_model.predict(X_test)

In [ ]:
plot_regression_diagnostic_plots(y_train, y_pred_train, y_test, y_pred_test)

In [ ]:
X, y = df_filter[['PM2.5', 'PM10']], df_filter['AQI']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, shuffle=False)
model = XGBRegressor(n_estimators=500, alpha = 100, learning_rate = 0.1)
model.fit(X_train, y_train)
model.score(X_test, y_test)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
plot_regression_diagnostic_plots(y_train, y_pred_train, y_test, y_pred_test)

In [ ]:
def df_split_scale(df, scaler='ss', test_size=0.3, shuffle=True, scale=False):
    """
    @params:
    df: input dataframe
    scaler: 'ss' = StandardScaler(), 'mm' = MinMaxScaler()
    scale: (by Default = False meaning no scaling)

    Scales the train and test df and split them into
    X_train, X_test, y_train, y_test
    """
    # Splitting the data
    X, y = df[['PM2.5', 'PM10']], df['AQI']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, shuffle=shuffle)

    if scale == True: # scaling
        if scaler == 'ss':
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            return X_train_scaled, X_test_scaled, y_train, y_test, scaler
        elif scaler == 'mm':
            scaler = MinMaxScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            return X_train_scaled, X_test_scaled, y_train, y_test, scaler
        else:
            return X_train, X_test, y_train, y_test, None
    else:  # no scaling
        return X_train, X_test, y_train, y_test, None

Before training model actually lets experiment a bit on scaled and non-scaled data to see how they performs. Meanwhile, doing these we are going to check cross_validation scores and also check the kdeplot of the test and predicted data


In [ ]:
def preprocess_plot_score(df, test_size=0.3, model=LinearRegression(), shuffle=True, scaler='ss', scale=False):
    """
    @params:
    df: input dataframe
    test_size: (by Default = 0.3)
    model: (by Default = LinearRegression())
    scale: (by Default = 'False')

    This function can be used train and test models
    validation scores to see which models are performing
    on data on specific scaling options used
    """

    # Splitting the data and scaling
    X_train, X_test, y_train, y_test, _ = df_split_scale(df, scaler=scaler, test_size=test_size, shuffle=shuffle, scale=scale)

    # Training the model
    model = model
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Cross-validation scores
    cv_scores = -1 * cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')

    # Plotting KDE with axes and cross-validation scores
    plot_regression_diagnostic_plots(y_train, y_train_pred, y_test, y_test_pred)
    print(cv_scores.mean())

    # Evaluate the model
    train_score = 1 * mean_squared_error(y_train, y_train_pred)
    test_score = 1 * mean_squared_error(y_test, y_test_pred)

    return model, train_score, test_score, cv_scores

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, scaler='mm', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model=RandomForestRegressor(), scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

# Modelling and Hypertuning

Following models are selection for training

1. SGDRegressor
2. XGBoostRegressor
3. RandomForestRegressor
4. LBGMRegressor
5. SGDRegressor

In this part, we are going train models, hypertune them and also see best performing model evaluation of neg_mean_squared_error, r2_score. Also, plot the best performing model on the basis of hypertuned parameters.

1. Model Selection
2. Model Training
3. Model Evaluation
4. Model Hypertuning


In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model= ExtraTreesRegressor(), scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
from sklearn.linear_model import ElasticNet, Lasso, Ridge, BayesianRidge
from sklearn.linear_model import OrthogonalMatchingPursuit, Lars

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model=ElasticNet())
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model= Lasso())
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model=BayesianRidge(), scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model=OrthogonalMatchingPursuit(), scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory=False)
model, train_score, test_score, cv_scores = preprocess_plot_score(df_filter, model= Lars(), scaler='ss', scale=True)
print("Training Score:", train_score)
print("Testing Score:", test_score)
print("Cross_Validation Score:", cv_scores)

In [ ]:
df_filter = pd.read_csv('clean_filtered_data.csv', low_memory = False)
X_train, X_test, y_train, y_test, scaler = df_split_scale(df_filter, scaler='ss', test_size=0.3, shuffle=True, scale=True)

In [ ]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [ ]:
def hyper_tune_regression_model(X_train, y_train, model_params, scoring='neg_mean_squared_error', cv=5, n_iter=30, random_state=None):
    """
    Hyper-tunes a regression model using RandomizedSearchCV.

    Parameters:
    X_train: array-like, shape (n_samples, n_features)
        Training data.
    y_train: array-like, shape (n_samples,)
        Target values.
    model: Estimator object.
        The regression model to be hyper-tuned.
    param_grid: dict
        Dictionary with parameters names (string) as keys and distributions or lists of parameters to try.
    scoring: str or callable, default='neg_mean_squared_error'
        A scoring strategy to evaluate the performance of the cross-validated model.
    cv: int, default=5
        Number of cross-validation folds.
    n_iter: int, default=10
        Number of parameter settings that are sampled.
    random_state: int or None, default=None
        Controls the random seed for reproducibility.

    Returns:
    best_model: Estimator object
        The best hyper-tuned model.
    best_params: dict
        The best hyperparameters found during hyperparameter tuning.
    best_score: float
        The scoring of the best hyper-tuned model.
    """
    model = model_params['model']
    param_grid = model_params['param_grid']
    # Instantiate RandomizedSearchCV
    random_search = RandomizedSearchCV(model,
                                       param_distributions=param_grid,
                                       n_iter=n_iter,
                                       cv=cv,
                                       scoring=scoring,
                                       random_state=random_state,
                                       n_jobs = -1)

    # Fit RandomizedSearchCV to the training data
    random_search.fit(X_train, y_train)

    # Get the best model, best parameters, and best score
    best_model = random_search.best_estimator_
    best_params = random_search.best_params_
    best_score = random_search.best_score_

    # Print the scoring of the best model
    print(f"Best Score: {best_score:.4f}")

    return (best_model, best_params, best_score)


In [ ]:
from sklearn.linear_model import ElasticNet, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor

model_params = {
    'ElasticNet': {
        'model': ElasticNet(),
        'param_grid': {
            'alpha': [0.001, 0.01, 0.1, 1, 10],
            'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
            'fit_intercept': [True, False],
            'max_iter': [1000, 2000, 3000],
            'selection': ['cyclic', 'random']
        }
    },
    'SGDRegressor': {
        'model': SGDRegressor(),
        'param_grid': {
            'alpha': [0.0001, 0.001, 0.01],
            'l1_ratio': [0, 0.25, 0.5, 0.75, 1],
            'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'max_iter': [1000, 2000, 3000],
            'tol': [1e-3, 1e-4, 1e-5]
        }
    },
    'KNeighborsRegressor': {
        'model': KNeighborsRegressor(),
        'param_grid': {
            'n_neighbors': [3, 5, 7, 9],
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
            'leaf_size': [30, 40, 50],
            'p': [1, 2]
        }
    }
}

In [ ]:
best_models = []
best_models.append(hyper_tune_regression_model(X_train, y_train, model_params['ElasticNet']))
best_models.append(hyper_tune_regression_model(X_train, y_train, model_params['SGDRegressor']))
best_models.append(hyper_tune_regression_model(X_train, y_train, model_params['KNeighborsRegressor']))

In [ ]:
best_models

In [ ]:
# Initialize lists to store results for each model
results = []

# Train and evaluate each model
for i, (best_model, _, _) in enumerate(best_models):
    # Train the model with best parameters
    best_model.fit(X_train, y_train)
    model_filename = f"model_no_{i + 1}.pkl"

    # Serialize the model to disk
    with open(model_filename, "wb") as f:
        pickle.dump(best_model, f)

    # Make predictions on test data
    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    # Calculate residuals
    residuals = y_test - y_pred_test

    # Calculate R^2 score
    # r2 = r2_score(y_test, y_pred)

    # Calculate mean squared error
    mse = mean_squared_error(y_test, y_pred_test)

    # Store results for the current model
    results.append({
        'model': best_model,
        'y_true': y_test,
        'y_pred_train': y_pred_train,
        'y_pred_test': y_pred_test,
        'residuals': residuals,
        # 'r2_score': r2,
        'mse': mse
    })

# Print test scores for each model
for i, result in enumerate(results):
    print(f"Model {i+1}:")
    # print(f"  R^2 Score: {result['r2_score']:.4f}")
    print(f"  Mean Squared Error: {result['mse']:.4f}")

In [ ]:
plot_regression_diagnostic_plots(y_train, results[0]['y_pred_train'], y_test, results[0]['y_pred_test'])

In [ ]:
plot_regression_diagnostic_plots(y_train, results[1]['y_pred_train'], y_test, results[1]['y_pred_test'])

In [ ]:
plot_regression_diagnostic_plots(y_train, results[2]['y_pred_train'], y_test, results[2]['y_pred_test'])

# Model Interpretation


In [ ]:
# Train the Random Forest Regressor model
model = pickle.load(open('model_no_3.pkl', 'rb'))
model.fit(X_train, y_train)

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import numpy as np

# Get permutation importance
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)

# Get the names of the features
feature_names = ['PM2.5', 'PM10']

# Sort the features based on their importance scores
sorted_indices = np.argsort(result.importances_mean)
sorted_feature_importances = result.importances_mean[sorted_indices]
sorted_feature_names = [feature_names[i] for i in sorted_indices]

# Plot feature importances
plt.figure(figsize=(10, 6))
plt.barh(sorted_feature_names, sorted_feature_importances)
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.title('Permutation Feature Importance')
plt.show()

In [ ]:
# Use permutation importance scores instead of feature_importances_
feature_importances = result.importances_mean  # This replaces model.feature_importances_

# Compute contributions for each test instance
contributions = X_test * feature_importances  # Element-wise multiplication

# Get predicted values for test set
predictions = model.predict(X_test)

# Compute contributions for the training set
train_contributions = X_train * feature_importances  # Element-wise multiplication

# Get predictions for the training set
train_predictions = model.predict(X_train)